In [ ]:
!uv pip install datasets plotly pillow numpy pandas matplotlib tqdm opencv-python

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt
from datasets import load_dataset
from PIL import Image, ImageStat

from datasets import load_dataset
dataset = load_dataset("1aurent/ADE20K", split="train",num_samples)
sample_size = min(1000, len(dataset))
ds_sample = dataset.select(range(sample_size))

print(f"Total Dataset Size: {len(dataset)}")
print(f"Sampled for EDA: {sample_size} images")

In [ ]:
from tqdm.auto import tqdm
import cv2
image_stats = []

for idx, item in enumerate(tqdm(ds_sample, desc="Extracting Image Metrics")):
    img = item['image']
    img_arr = np.array(img)
    
    w, h = img.size
    aspect_ratio = w / h
    
    if len(img_arr.shape) == 3:
        gray = cv2.cvtColor(img_arr, cv2.COLOR_RGB2GRAY)
        r_mean, g_mean, b_mean = img_arr[:,:,0].mean(), img_arr[:,:,1].mean(), img_arr[:,:,2].mean()
    else:
        gray = img_arr
        r_mean, g_mean, b_mean = gray.mean(), gray.mean(), gray.mean()

    brightness = gray.mean()
    contrast = gray.std()
    sharpness = cv2.Laplacian(gray, cv2.CV_64F).var()
    
    image_stats.append({
        'id': idx, 'width': w, 'height': h, 'aspect_ratio': aspect_ratio,
        'brightness': brightness, 'contrast': contrast, 'sharpness': sharpness,
        'R': r_mean, 'G': g_mean, 'B': b_mean
    })

df_images = pd.DataFrame(image_stats)
print(f"Dataset Size: {len(dataset)} images (Analyzed {sample_size} images)")

In [ ]:
# Width vs Height Scatter with Marginals
fig_dims = px.scatter(df_images, x="width", y="height", 
                      marginal_x="histogram", marginal_y="histogram",
                      title="Size Marginal Distribution (Width vs Height)",
                      color_discrete_sequence=['#3b82f6'])
fig_dims.show()

# Aspect Ratio Distribution
fig_ar = px.histogram(df_images, x="aspect_ratio", nbins=30,
                      title="Aspect Ratio Distribution",
                      color_discrete_sequence=['#10b981'])
fig_ar.add_vline(x=1.0, line_dash="dash", line_color="red", annotation_text="Square (1:1)")
fig_ar.show()

In [ ]:
from plotly.subplots import make_subplots

fig_qual = make_subplots(rows=1, cols=3, subplot_titles=("Brightness", "Contrast", "Sharpness"))

fig_qual.add_trace(go.Histogram(x=df_images['brightness'], name="Brightness", marker_color='#f59e0b'), row=1, col=1)
fig_qual.add_trace(go.Histogram(x=df_images['contrast'], name="Contrast", marker_color='#8b5cf6'), row=1, col=2)
fig_qual.add_trace(go.Histogram(x=df_images['sharpness'], name="Sharpness", marker_color='#ef4444'), row=1, col=3)

fig_qual.update_layout(title_text="Image Quality Metrics Distribution", showlegend=False)
fig_qual.show()

# RGB 3D Scatter (lấy mẫu 200 điểm để render mượt)
df_rgb_sample = df_images.sample(min(200, len(df_images)))
fig_rgb = px.scatter_3d(df_rgb_sample, x='R', y='G', z='B',
                        color=df_rgb_sample.apply(lambda r: f"rgb({int(r.R)},{int(r.G)},{int(r.B)})", axis=1),
                        title="RGB Color Space Distribution (Mean per Image)")
fig_rgb.show()

In [ ]:
seg_stats = []
global_class_counts = np.zeros(151) 

for idx, item in enumerate(tqdm(ds_sample, desc="Extracting Mask Metrics")):
    mask_img = item['segmentations'][0] 
    mask_arr = np.array(mask_img)
    
    if len(mask_arr.shape) == 3:
        mask_arr = mask_arr[:, :, 0]
        
    unique_classes, counts = np.unique(mask_arr, return_counts=True)
    
    for cls, count in zip(unique_classes, counts):
        if cls <= 150:
            global_class_counts[cls] += count
            
    num_unique_classes = len(unique_classes[unique_classes > 0])
    unlabeled_ratio = (counts[unique_classes == 0][0] / mask_arr.size) if 0 in unique_classes else 0
    
    seg_stats.append({
        'id': idx,
        'num_objects': num_unique_classes,
        'unlabeled_ratio': unlabeled_ratio
    })

df_seg = pd.DataFrame(seg_stats)

In [ ]:
# Distribution of unique classes per image
fig_obj = px.histogram(df_seg, x="num_objects", nbins=20,
                       title="Number of Unique Classes per Image",
                       color_discrete_sequence=['#ec4899'])
fig_obj.show()

# Top 20 Global Classes Distribution (By Pixel Area)
top_classes_idx = global_class_counts.argsort()[::-1][1:21] # Bỏ class 0 (background)
top_classes_counts = global_class_counts[top_classes_idx]

df_top_classes = pd.DataFrame({
    'Class ID': [f"Class {i}" for i in top_classes_idx],
    'Pixel Count': top_classes_counts
})

fig_cls = px.bar(df_top_classes, x='Pixel Count', y='Class ID', orientation='h',
                 title="Top 20 Most Frequent Classes (by Total Pixels)",
                 color='Pixel Count', color_continuous_scale='Viridis')
fig_cls.update_layout(yaxis={'categoryorder':'total ascending'})
fig_cls.show()

In [ ]:
def show_gallery(dataset, num_samples=4):
    np.random.seed(42)
    indices = np.random.choice(len(dataset), num_samples, replace=False)
    
    fig, axes = plt.subplots(num_samples, 3, figsize=(15, 5 * num_samples))
    plt.suptitle("📸 Segmentation Gallery (Original | Mask | Overlay)", fontsize=16, y=0.92)
    
    for i, idx in enumerate(indices):
        item = dataset[int(idx)]
        img = np.array(item['image'])
        
        # Fix: Trỏ đúng vào ảnh đầu tiên trong list 'segmentations'
        mask_arr = np.array(item['segmentations'][0])
        if len(mask_arr.shape) == 3:
            mask_arr = mask_arr[:, :, 0]
        
        # Colorize mask
        mask_colored = plt.cm.get_cmap('tab20')(mask_arr / 150.0)[:, :, :3]
        
        # Overlay
        overlay = (img / 255.0) * 0.5 + mask_colored * 0.5
        
        axes[i, 0].imshow(img)
        axes[i, 0].set_title(f"Image {idx}")
        axes[i, 0].axis('off')
        
        axes[i, 1].imshow(mask_arr, cmap='tab20', vmin=0, vmax=150)
        axes[i, 1].set_title("Segmentation Mask")
        axes[i, 1].axis('off')
        
        axes[i, 2].imshow(overlay)
        axes[i, 2].set_title("Overlay (50% Opacity)")
        axes[i, 2].axis('off')
        
    plt.tight_layout()
    plt.show()

show_gallery(ds_sample, num_samples=4)